# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR^2 dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. These are the main data structures in a Croissant dataset.

We'll extract all record sets and their fields, referencing only by their `@id` values for consistency.

In [ ]:
# List all RecordSets and their Fields by @id
record_sets = dataset.metadata.recordSet
if record_sets:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        fields = rs.get('field', [])
        for field in fields:
            print(f"  Field @id: {field['@id']}")

    # Example: Show a sample from one RecordSet
    for rs in record_sets:
        print(f"\nSample records from RecordSet @id: {rs['@id']}")
        for x in dataset.records(record_set=rs['@id']):
            print(x)
            break  # Show only one sample record from each RecordSet
else:
    print("No record sets found in metadata.")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.
We use the record set and field `@id`s from the previous section. All entities are referenced by `@id` only.

In [ ]:
# Prepare list of RecordSet @id values
record_sets = dataset.metadata.recordSet
record_set_ids = [rs['@id'] for rs in record_sets] if record_sets else []

# Load data from each RecordSet into a DataFrame
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded DataFrame for RecordSet @id: {rs_id}")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head(), "\n")
    else:
        print(f"No records found for RecordSet @id: {rs_id}")

# Choose first available RecordSet for further analysis
if record_set_ids:
    primary_record_set_id = record_set_ids[0]
    print(f"Using RecordSet @id: {primary_record_set_id}")
else:
    primary_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
We will apply data processing steps such as filtering records, normalizing numeric fields, and grouping by categorical fields.

All fields and columns are referenced strictly by their `@id` identifier.

In [ ]:
# Example: Identify a numeric field @id for EDA
if primary_record_set_id:
    df = dataframes[primary_record_set_id]
    # Attempt to identify numeric fields by their @id (column names)
    numeric_fields = [col for col in df.columns if df[col].dtype in ['int64', 'float64']]
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        print(f"Using numeric field for analysis: {numeric_field_id}")
        threshold = 10  # Example threshold
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalization
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized field {numeric_field_id}:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Find a categorical/grouping field @id
        group_fields = [col for col in df.columns if df[col].dtype == 'object']
        group_field_id = group_fields[0] if group_fields else None
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped by {group_field_id}:")
            print(grouped_df.head())
        else:
            print("No categorical field found for grouping.")
    else:
        print("No numeric fields found for EDA.")
else:
    print("No main RecordSet available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. All references are by `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the numeric field distribution
if primary_record_set_id and numeric_fields:
    df = dataframes[primary_record_set_id]
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id} (@id)")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If grouping/categorical field is available
    if group_field_id:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id} (@id)")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the FAIR^2 dataset exploration.

This notebook demonstrates loading, overview, extraction, and analysis of the dataset using the `mlcroissant` library, referencing all entities strictly by their `@id` values for reproducibility and schema-compliance.